In [23]:
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

### Load Pdf

In [5]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()
len(docs)

10

#### Splitting pdf to chunks

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(docs)
len(splitter_data)

10

#### Covert the chunks to Embedding

In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

#### Store Embedding in Vector DB

In [ ]:
vector_stroe = Chroma.from_documents(
    documents=splitter_data,
    embedding=embeddings
    
)

In [12]:
query = "Machine Learnings and Data Science Content"
data = vector_stroe.similarity_search(query=query, k=2)
data[0]

Document(id='7b2bc991-2f1c-4ded-958f-daf9832d04f2', metadata={'moddate': '2025-12-13T08:07:26+00:00', 'title': '📘 1-Year Roadmap: Data Analytics, Data Science & GenAI', 'page_label': '10', 'producer': 'Skia/PDF m137', 'source': '../data/data_science_syllabus.pdf', 'total_pages': 10, 'page': 9, 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'creationdate': '2025-12-13T08:07:26+00:00'}, page_content='📘  1\ue088Year Roadmap: Data Analytics, Data Science & GenAI\n10')

In [13]:
Context = ""
for doc in data:
    Context += doc.page_content + "\n"
print(Context)

📘  1Year Roadmap: Data Analytics, Data Science & GenAI
10
Customer Segmentation
E-commerce Product Clustering
🎤  Mock Interview: Unsupervised Learning  Features
✅  Module 8: Time Series + Intro to Deep Learning (3 
weeks)
Duration: Month 8
Topics:
Time Series components, lag, rolling stats
Forecasting: ARIMA, Prophet
Intro to Neural Networks
Perceptron, basic Keras model
Tools:
Statsmodels, Prophet, Keras, TensorFlow
Mini Project:
Stock Price or Weather Forecasting
🎤  Mock Interview: Forecasting  Neural Net Basics
✅  Module 8.5: Big Data with PySpark (2 weeks)
Duration: Between Month 89
Topics:
What is Big Data?
PySpark architecture & setup
DataFrames, RDDs, Transformations, Actions
PySpark SQL  basic MLlib
Databricks (optional)
Tools:
PySpark, Google Colab Findspark), Databricks
📘  1Year Roadmap: Data Analytics, Data Science & GenAI
6



In [22]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


#### Chain  - Context_generate | prompt | llm | strparser

In [25]:
def get_context(query:str):
    data = vector_stroe.similarity_search(query=query, k=2)
    Context = ""
    for doc in data:
        Context += doc.page_content + "\n"
    return {
        "context": Context,
        "question": query
    }

In [26]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant and provide answer based on the context for user Query.
If you don't know the answer, just say "I don't know". Do not try to make up an answer.
Context: {context}
Question: {question}
""")

In [27]:
rag_chain = get_context | prompt |llm 

In [31]:
res = rag_chain.invoke("What is the Java course?")
print(res.content[0]["text"])

I don't know.
